In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from scipy.linalg import solve_discrete_are, solve_discrete_lyapunov
from scipy.spatial import cKDTree
import warnings
import matplotlib.gridspec as gridspec
import matplotlib.colors as mcolors
from matplotlib.lines import Line2D
warnings.filterwarnings('ignore')

plt.rcParams.update({
    'font.family': 'serif',
    'font.serif': ['Times New Roman', 'Computer Modern Roman', 'serif'],
    'mathtext.fontset': 'cm',
    'font.size': 18,            
    'axes.labelsize': 20, 
    'axes.titlesize': 18,
    'figure.titlesize': 22,
    'legend.fontsize': 14,
    'xtick.labelsize': 16,
    'ytick.labelsize': 16,
    'pdf.fonttype': 42,
    'ps.fonttype': 42
})

# 6-DOF Quadcopter Dynamics & LQR Solvers

In [ ]:
def get_drone_dynamics(dt=0.05):
    """
    Returns the discretized A and B matrices for a 6-DOF Quadcopter.
    States (12): [roll, pitch, yaw, X, Y, Z, p, q, r, u, v, w]
    Actions (4): [Thrust, Torque-X, Torque-Y, Torque-Z]
    Derived from: '(LQR)6 DOF Quadcoptor' Physical Model
    """
    Ac = np.zeros((12, 12))
    Ac[0, 6] = 1.0; Ac[1, 7] = 1.0; Ac[2, 8] = 1.0
    Ac[3, 9] = 1.0; Ac[4, 10] = 1.0; Ac[5, 11] = 1.0
    Ac[9, 1] = -9.8; Ac[10, 0] = 9.8

    Bc = np.zeros((12, 4))
    Bc[6, 1] = 10.0      # Torque-X -> p_dot
    Bc[7, 2] = 10.0      # Torque-Y -> q_dot
    Bc[8, 3] = 6.6667    # Torque-Z -> r_dot
    Bc[11, 0] = 5.0      # Thrust -> w_dot (Z acceleration)

    A_d = np.eye(12) + Ac * dt
    B_d = Bc * dt
    return A_d, B_d

def generate_expert_rollouts(N_samples, K, A, B, noise_std=0.1):
    """Generates continuous-state dataset by unrolling a closed-loop LQR controller."""
    state_dim, action_dim = K.shape[1], K.shape[0]
    X_data, U_data = np.zeros((N_samples, state_dim)), np.zeros((N_samples, action_dim))
    
    x_t = np.random.randn(state_dim) * 2.0 
    
    for t in range(N_samples):
        u_t = -K @ x_t + np.random.randn(action_dim) * noise_std 
        X_data[t, :] = x_t
        U_data[t, :] = u_t
        
        x_t = A @ x_t + B @ u_t
        x_t += np.random.randn(state_dim) * 0.05

    return X_data, U_data

def compute_policy_cost(K_hat, Q, R, A, B):
    """Evaluates the infinite-horizon cost of a learned feedback matrix K_hat."""
    A_cl = A - B @ K_hat
    if np.max(np.abs(np.linalg.eigvals(A_cl))) >= 1.0:
        return np.nan # Policy is unstable
    Q_cl = Q + K_hat.T @ R @ K_hat
    try:
        P_hat = solve_discrete_lyapunov(A_cl.T, Q_cl)
        return np.trace(P_hat)
    except:
        return np.nan

def get_pareto_front(A, B, Q1, Q2, R, num_points=200):
    """Computes the continuous optimal frontier between two LQR objective weights."""
    print(f"Computing theoretical LQR Pareto front ({num_points} points)...")
    pf_costs = []
    weights = np.linspace(0, 1, num_points)
    
    for w in weights:
        Qw = w * Q1 + (1 - w) * Q2
        Pw = solve_discrete_are(A, B, Qw, R)
        Kw = np.linalg.inv(R + B.T @ Pw @ B) @ (B.T @ Pw @ A)
        
        J1 = compute_policy_cost(Kw, Q1, R, A, B)
        J2 = compute_policy_cost(Kw, Q2, R, A, B)
        pf_costs.append([J1, J2])
        
    return np.array(pf_costs)

def get_linf_distance_to_pf(K_hat, pf_costs, Q1, Q2, R, A, B):
    """Measures Chebyshev L-infinity suboptimality against the continuous frontier."""
    J1_hat = compute_policy_cost(K_hat, Q1, R, A, B)
    J2_hat = compute_policy_cost(K_hat, Q2, R, A, B)
    
    if np.isnan(J1_hat) or np.isnan(J2_hat):
        return np.nan
        
    J_hat = np.array([J1_hat, J2_hat])
    distances = np.max(np.abs(pf_costs - J_hat), axis=1)
    return np.min(distances)

# Imitation Learning Estimators (Independent vs MA-BC)


In [ ]:
def fit_independent(X, U, l2_reg=1e-3):
    """Behavioral Cloning via Ridge Regression for guaranteed LQR stability."""
    W = np.linalg.solve(X.T @ X + l2_reg * np.eye(X.shape[1]), X.T @ U)
    return -W.T

def fit_pooled(X1, U1, X2, U2, l2_reg=1e-3):
    """Naive Joint Dataset Pooling."""
    X_pool = np.vstack([X1, X2])
    U_pool = np.vstack([U1, U2])
    W = np.linalg.solve(X_pool.T @ X_pool + l2_reg * np.eye(X_pool.shape[1]), X_pool.T @ U_pool)
    return -W.T

def get_augmented_dataset_minmax(X_base, U_base, X_other, U_other, delta_state, K_base, optimistic=False):
    """
    Corrected MA-BC: Ensures that at delta_s=0, no data is shared.
    Applies a dynamic upper bound that scales the Lipschitz constant 
    by the actual state distance to prevent excessively generous thresholds.
    """
    if delta_state <= 1e-9:
        return X_base, U_base

    combined_s = np.vstack([X_base, X_other])
    s_min, s_max = np.min(combined_s, axis=0), np.max(combined_s, axis=0)
    s_range = np.clip(s_max - s_min, 1e-8, None)
    
    K_norm = K_base * s_range 
    L_norm_dim = np.linalg.norm(K_norm, axis=1, ord=2)
    
    
    norm_base = (X_base - s_min) / s_range
    norm_other = (X_other - s_min) / s_range
    
    tree_base = cKDTree(norm_base)
    all_neighbors = tree_base.query_ball_point(norm_other, r=delta_state, workers=-1)
    
    shared_x_indices = []
    for i, neighbor_indices in enumerate(all_neighbors):
        if len(neighbor_indices) > 0:
            a_base_neighbors = U_base[neighbor_indices]
            
            actual_state_distances = np.linalg.norm(norm_base[neighbor_indices] - norm_other[i], axis=1)
            
            dynamic_action_bound = L_norm_dim * actual_state_distances[:, np.newaxis]
            
            action_diffs_abs = np.abs(a_base_neighbors - U_other[i])
            is_match = np.all(action_diffs_abs <= dynamic_action_bound, axis=1)
            
            if np.any(is_match):
                shared_x_indices.append(i)
        else:
            if optimistic:
                shared_x_indices.append(i) 
            
    if shared_x_indices:
        X_aug = np.vstack([X_base, X_other[shared_x_indices]])
        U_aug = np.vstack([U_base, U_other[shared_x_indices]])
    else:
        X_aug, U_aug = X_base, U_base
        
    return X_aug, U_aug

def fit_threshold_mabc(X1, U1, X2, U2, d_s, K1, K2, optimistic=False):
    X1_aug, U1_aug = get_augmented_dataset_minmax(X1, U1, X2, U2, d_s, K_base=K1, optimistic=optimistic)
    X2_aug, U2_aug = get_augmented_dataset_minmax(X2, U2, X1, U1, d_s, K_base=K2, optimistic=optimistic)
    return fit_independent(X1_aug, U1_aug), fit_independent(X2_aug, U2_aug)

# Experiment Execution: Drone Multi-Objective Tradeoff


In [ ]:
def run_drone_sweep(pf_costs, A, B, Q1, Q2, R, 
                    N_list=[15, 20, 30, 50, 100, 200, 500], 
                    delta_s_list=[0.05, 0.1, 0.5]):
    
    P1_opt = solve_discrete_are(A, B, Q1, R)
    K1_true = np.linalg.inv(R + B.T @ P1_opt @ B) @ (B.T @ P1_opt @ A)
    P2_opt = solve_discrete_are(A, B, Q2, R)
    K2_true = np.linalg.inv(R + B.T @ P2_opt @ B) @ (B.T @ P2_opt @ A)

    results_k1 = {rf'MA-BC ($\delta_s={ds}$)': [] for ds in delta_s_list}
    results_k2 = {rf'MA-BC ($\delta_s={ds}$)': [] for ds in delta_s_list}
    results_k1.update({'Independent': [], 'Naive Pool': []})
    results_k2.update({'Independent': [], 'Naive Pool': []})
    
    print(f"Evaluating Sample Complexity across {len(N_list)} sizes...")
    
    for N in N_list:
        gaps_k1 = {k: [] for k in results_k1.keys()}
        gaps_k2 = {k: [] for k in results_k2.keys()}
        
        for seed in range(10): 
            np.random.seed(seed + N*100)
            X1, U1 = generate_expert_rollouts(N, K1_true, A, B)
            X2, U2 = generate_expert_rollouts(N, K2_true, A, B)

            k1_ind = fit_independent(X1, U1)
            k2_ind = fit_independent(X2, U2)
            k_pool = fit_pooled(X1, U1, X2, U2)

            gaps_k1['Independent'].append(get_linf_distance_to_pf(k1_ind, pf_costs, Q1, Q2, R, A, B))
            gaps_k2['Independent'].append(get_linf_distance_to_pf(k2_ind, pf_costs, Q1, Q2, R, A, B))
            gaps_k1['Naive Pool'].append(get_linf_distance_to_pf(k_pool, pf_costs, Q1, Q2, R, A, B))
            gaps_k2['Naive Pool'].append(get_linf_distance_to_pf(k_pool, pf_costs, Q1, Q2, R, A, B))
            
            for ds in delta_s_list:
                key = rf'MA-BC ($\delta_s={ds}$)'
                K1_hat, K2_hat = fit_threshold_mabc(X1, U1, X2, U2, ds, K1_true, K2_true, optimistic=False)
                gaps_k1[key].append(get_linf_distance_to_pf(K1_hat, pf_costs, Q1, Q2, R, A, B))
                gaps_k2[key].append(get_linf_distance_to_pf(K2_hat, pf_costs, Q1, Q2, R, A, B))

        for k in results_k1.keys():
            results_k1[k].append(np.nanmedian(gaps_k1[k]))
            results_k2[k].append(np.nanmedian(gaps_k2[k]))

    return results_k1, results_k2

A, B = get_drone_dynamics()
R = np.eye(4) # Control cost base

Q_tracking = np.diag([10, 10, 10, 100, 100, 100, 1, 1, 1, 1, 1, 1])
Q_eco = np.diag([0.1, 0.1, 0.1, 1, 1, 1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1])

alpha = 0.5

w1_track = 0.5 + (0.5 * alpha)
w1_eco = 0.5 - (0.5 * alpha)

w2_track = 0.5 - (0.5 * alpha)
w2_eco = 0.5 + (0.5 * alpha)

Q1 = (w1_track * Q_tracking) + (w1_eco * Q_eco) # Agile Expert
Q2 = (w2_track * Q_tracking) + (w2_eco * Q_eco) # Eco Expert

pf_costs = get_pareto_front(A, B, Q1, Q2, R)

delta_sweeps = np.linspace(0, 1, 50)
n_sizes = [10, 20, 30, 40, 50, 60, 70, 80, 90, 100, 200, 300, 400, 500, 600, 700, 800, 900, 1000]

res_k1, res_k2 = run_drone_sweep(pf_costs, A, B, Q1, Q2, R, n_sizes, delta_sweeps)

# Plotting


In [ ]:
def plot_final_learning_curves(n_sizes, res_k1, res_k2, delta_s_list, filename="drone_learning_curve.pdf"):
    fig = plt.figure(figsize=(10, 6.0))
    
    gs = gridspec.GridSpec(1, 3, width_ratios=[1, 1, 0.05], wspace=0.25)
    ax1 = fig.add_subplot(gs[0])
    ax2 = fig.add_subplot(gs[1])
    cax = fig.add_subplot(gs[2])

    cmap = cm.viridis
    norm = mcolors.Normalize(vmin=min(delta_s_list), vmax=max(delta_s_list))
    sm = cm.ScalarMappable(cmap=cmap, norm=norm); sm.set_array([])

    for ax_idx, (ax, res) in enumerate(zip([ax1, ax2], [res_k1, res_k2])):
        for ds in delta_s_list:
            key = rf'MA-BC ($\delta_s={ds}$)'
            if key in res:
                ax.plot(n_sizes, res[key], color=cmap(norm(ds)), linewidth=1.5, alpha=0.3, zorder=2)
        
        ax.plot(n_sizes, res['Naive Pool'], label='Naive BC', color='#d62728', 
                linestyle='--', linewidth=2.5, marker='X', markersize=6, zorder=10)
        ax.plot(n_sizes, res['Independent'], label='Isolated BC', color='orange', 
                linestyle='--', linewidth=2.5, marker='o', markersize=6, zorder=10)
        
        ax.set_yscale('log'); ax.set_xscale('log'); ax.grid(True, ls=":", alpha=0.6)
        ax.set_title('Agile Expert' if ax_idx == 0 else 'Economic Expert')
        if ax_idx == 0: 
            ax.set_ylabel(r"$L_\infty$ Distance to Pareto Front")
            
    fig.colorbar(sm, cax=cax).set_label(r'Sharing Threshold ($\delta_s$)')
    ax1.legend(loc='upper right', framealpha=0.9, fontsize=10)
    
    fig.text(0.48, 0.20, r"Dataset Size per Expert ($N$ Trajectories)", ha='center')
    
    fig.subplots_adjust(left=0.08, right=0.88, bottom=0.30, top=0.90)
    
    plt.savefig(filename, dpi=300)
    plt.show()

In [ ]:
plot_final_learning_curves(n_sizes, res_k1, res_k2, delta_s_list=delta_sweeps)

# Visualizing bias and variance trade-off

In [ ]:
def plot_empirical_lqr_tradeoff(pf_costs, A, B, Q1, Q2, R, K1_true, K2_true, N=100):
    print(f"Running empirical bias-variance sweep at N={N}...")
    
    np.random.seed(42)
    X1, U1 = generate_expert_rollouts(N, K1_true, A, B)
    X2, U2 = generate_expert_rollouts(N, K2_true, A, B)
    
    K1_indep = fit_independent(X1, U1)
    err1_indep = get_linf_distance_to_pf(K1_indep, pf_costs, Q1, Q2, R, A, B)
    K2_indep = fit_independent(X2, U2)
    err2_indep = get_linf_distance_to_pf(K2_indep, pf_costs, Q1, Q2, R, A, B)
    
    K_naive = fit_pooled(X1, U1, X2, U2)
    err1_naive = get_linf_distance_to_pf(K_naive, pf_costs, Q1, Q2, R, A, B)
    err2_naive = get_linf_distance_to_pf(K_naive, pf_costs, Q1, Q2, R, A, B)
    
    delta_s_grid = np.logspace(-2, 0, 50) 
    results_ex1 = {'pct': [], 'bias': [], 'err': []}
    results_ex2 = {'pct': [], 'bias': [], 'err': []}
    
    for ds in delta_s_grid:
        X1_aug, U1_aug = get_augmented_dataset_minmax(X1, U1, X2, U2, ds, K_base=K1_true, optimistic=False)
        num_pooled1 = len(X1_aug) - len(X1)
        results_ex1['pct'].append((num_pooled1 / len(X2)) * 100.0)
        if num_pooled1 > 0: bias1 = np.mean(np.linalg.norm(U1_aug[len(U1):] - (-X1_aug[len(X1):] @ K1_true.T), axis=1))
        else: bias1 = 0.0
        results_ex1['bias'].append(bias1)
        K1_hat = fit_independent(X1_aug, U1_aug)
        err1 = get_linf_distance_to_pf(K1_hat, pf_costs, Q1, Q2, R, A, B)
        results_ex1['err'].append(err1 if not np.isnan(err1) else 1e4)

        X2_aug, U2_aug = get_augmented_dataset_minmax(X2, U2, X1, U1, ds, K_base=K2_true, optimistic=False)
        num_pooled2 = len(X2_aug) - len(X2)
        results_ex2['pct'].append((num_pooled2 / len(X1)) * 100.0)
        if num_pooled2 > 0: bias2 = np.mean(np.linalg.norm(U2_aug[len(U2):] - (-X2_aug[len(X2):] @ K2_true.T), axis=1))
        else: bias2 = 0.0
        results_ex2['bias'].append(bias2)
        K2_hat = fit_independent(X2_aug, U2_aug)
        err2 = get_linf_distance_to_pf(K2_hat, pf_costs, Q1, Q2, R, A, B)
        results_ex2['err'].append(err2 if not np.isnan(err2) else 1e4)
    
    
    fig = plt.figure(figsize=(10, 6.0))
    
    gs = gridspec.GridSpec(1, 3, width_ratios=[1, 1, 0.05], wspace=0.25)
    ax1 = fig.add_subplot(gs[0])
    ax2 = fig.add_subplot(gs[1])
    cax = fig.add_subplot(gs[2]) 
    cax.axis('off') 
    
    def plot_tradeoff_axis(ax, res, err_indep, err_naive, title, is_left_plot):
        ax_twin = ax.twinx()  
        ax.plot(delta_s_grid, res['err'], '-', color='black', linewidth=3.0, label=r'MA-BC Pareto Distance ($L_\infty$)')
        ax.axhline(err_indep, color='orange', linestyle='--', linewidth=2.5, label=r'Isolated BC Pareto Distance ($L_\infty$)')
        ax.axhline(err_naive, color='#d62728', linestyle=':', linewidth=3.0, label=r'Naive BC Pareto Distance ($L_\infty$)')
        ax.plot(delta_s_grid, res['bias'], '-.', color='#d62728', linewidth=2.5, label=r'Empirical Action Bias ($L_2$)')
        ax_twin.fill_between(delta_s_grid, 0, res['pct'], color='#1f77b4', alpha=0.15, label='% Data Pooled')
        ax_twin.plot(delta_s_grid, res['pct'], '-', color='#1f77b4', linewidth=2.0)
        
        ax.set_yscale('log'); ax.set_xscale('log'); ax.grid(True, ls=':', alpha=0.6)
        ax.set_title(title)
        
        if is_left_plot:
            ax.set_ylabel(r'Error Metrics (Log Scale)')
            ax_twin.tick_params(labelright=False)
        else:
            ax_twin.set_ylabel('% Other Expert Data Pooled', color='#1f77b4', rotation=270, labelpad=15)
            ax_twin.tick_params(axis='y', labelcolor='#1f77b4')
        ax_twin.set_ylim(0, 105)
        return ax_twin

    ax_twin1 = plot_tradeoff_axis(ax1, results_ex1, err1_indep, err1_naive, 'Agile Expert', True)
    ax_twin2 = plot_tradeoff_axis(ax2, results_ex2, err2_indep, err2_naive, 'Economic Expert', False)

    handles, labels = [], []
    for ax in [ax1, ax_twin1]:
        for handle, label in zip(*ax.get_legend_handles_labels()):
            if label not in labels:
                handles.append(handle); labels.append(label)

    fig.text(0.48, 0.20, r'Sharing Threshold ($\delta_s$)', ha='center')

    fig.legend(handles, labels, loc='lower center', ncol=3, bbox_to_anchor=(0.48, 0.02), 
               framealpha=0.9, columnspacing=1.5)
    
    fig.subplots_adjust(left=0.08, right=0.88, bottom=0.30, top=0.90)
    
    plt.savefig('drone_lqr_data_bias.pdf', dpi=300) 
    plt.show()

In [ ]:
P1_opt = solve_discrete_are(A, B, Q1, R)
K1_true = np.linalg.inv(R + B.T @ P1_opt @ B) @ (B.T @ P1_opt @ A)

P2_opt = solve_discrete_are(A, B, Q2, R)
K2_true = np.linalg.inv(R + B.T @ P2_opt @ B) @ (B.T @ P2_opt @ A)

plot_empirical_lqr_tradeoff(
    pf_costs=pf_costs, 
    A=A, B=B, 
    Q1=Q1, Q2=Q2, R=R, 
    K1_true=K1_true, 
    K2_true=K2_true, 
    N=100
)

# Agile Expert Only

In [ ]:
def get_agile_tradeoff_data(pf_costs, A, B, Q1, Q2, R, K1_true, K2_true, N=100):
    print(f"Running empirical bias-variance sweep for Agile Expert at N={N}...")
    np.random.seed(42)
    
    X1, U1 = generate_expert_rollouts(N, K1_true, A, B)
    X2, U2 = generate_expert_rollouts(N, K2_true, A, B)
    
    K1_indep = fit_independent(X1, U1)
    err1_indep = get_linf_distance_to_pf(K1_indep, pf_costs, Q1, Q2, R, A, B)
    
    K_naive = fit_pooled(X1, U1, X2, U2)
    err1_naive = get_linf_distance_to_pf(K_naive, pf_costs, Q1, Q2, R, A, B)
    
    delta_s_grid = np.logspace(-2, 0, 50) 
    results_ex1 = {'pct': [], 'bias': [], 'err': []}
    
    for ds in delta_s_grid:
        X1_aug, U1_aug = get_augmented_dataset_minmax(X1, U1, X2, U2, ds, K_base=K1_true, optimistic=False)
        num_pooled1 = len(X1_aug) - len(X1)
        results_ex1['pct'].append((num_pooled1 / len(X2)) * 100.0)
        
        if num_pooled1 > 0: 
            bias1 = np.mean(np.linalg.norm(U1_aug[len(U1):] - (-X1_aug[len(X1):] @ K1_true.T), axis=1))
        else: 
            bias1 = 0.0
            
        results_ex1['bias'].append(bias1)
        K1_hat = fit_independent(X1_aug, U1_aug)
        err1 = get_linf_distance_to_pf(K1_hat, pf_costs, Q1, Q2, R, A, B)
        results_ex1['err'].append(err1 if not np.isnan(err1) else 1e4)

    return delta_s_grid, results_ex1, err1_indep, err1_naive

In [ ]:
def plot_agile_expert_summary(n_sizes, res_k1, delta_s_list, 
                              delta_s_grid, results_ex1, err1_indep, err1_naive, 
                              filename="drone_agile_summary.pdf"):
    fig = plt.figure(figsize=(16, 7.0))
    
    gs = gridspec.GridSpec(1, 4, width_ratios=[1, 0.03, 0.20, 1.15], wspace=0.15)
    ax1 = fig.add_subplot(gs[0])
    cax = fig.add_subplot(gs[1])
    ax2 = fig.add_subplot(gs[3])
    
    iso_style = {'color': 'orange', 'linestyle': '--', 'linewidth': 2.5, 'marker': 'o', 'markersize': 8}
    naive_style = {'color': '#d62728', 'linestyle': '--', 'linewidth': 2.5, 'marker': 'X', 'markersize': 8}
    
    cmap = cm.viridis
    norm = mcolors.Normalize(vmin=min(delta_s_list), vmax=max(delta_s_list))
    sm = cm.ScalarMappable(cmap=cmap, norm=norm); sm.set_array([])
    
    for ds in delta_s_list:
        key = rf'MA-BC ($\delta_s={ds}$)'
        if key in res_k1:
            ax1.plot(n_sizes, res_k1[key], color=cmap(norm(ds)), linewidth=1.5, alpha=0.4, zorder=2)
            
    ax1.plot(n_sizes, res_k1['Naive Pool'], zorder=10, **naive_style)
    ax1.plot(n_sizes, res_k1['Independent'], zorder=10, **iso_style)
    
    ax1.set_yscale('log')
    ax1.set_xscale('log')
    ax1.grid(True, ls=":", alpha=0.6)
    ax1.set_title('Sample Efficiency (Agile Expert)', fontsize=20, pad=15)
    
    ax1.set_ylabel(r"$L_\infty$ Distance to Pareto Front")
    ax1.set_xlabel(r"Dataset Size per Expert ($N$)")
    
    fig.colorbar(sm, cax=cax).set_label(r'Sharing Threshold ($\delta_s$)', labelpad=15)
    
    ax_twin = ax2.twinx()  
    
    ax2.plot(delta_s_grid, results_ex1['err'], '-', color='black', linewidth=3.0)
    
    ax2.plot(delta_s_grid, [err1_indep]*len(delta_s_grid), markevery=5, zorder=10, **iso_style)
    ax2.plot(delta_s_grid, [err1_naive]*len(delta_s_grid), markevery=5, zorder=10, **naive_style) 
    
    ax2.plot(delta_s_grid, results_ex1['bias'], '-.', color='#d62728', linewidth=2.5)
    
    ax_twin.fill_between(delta_s_grid, 0, results_ex1['pct'], color='#1f77b4', alpha=0.15)
    ax_twin.plot(delta_s_grid, results_ex1['pct'], '-', color='#1f77b4', linewidth=2.0)
    
    ax2.set_yscale('log')
    ax2.set_xscale('log')
    ax2.grid(True, ls=':', alpha=0.6)
    ax2.set_title('Empirical Bias-Data Tradeoff', fontsize=20, pad=15)
    ax2.set_xlabel(r'Sharing Threshold ($\delta_s$)')
    
    ax2.set_ylabel("Error Metric")
    
    ax_twin.set_ylabel('% Other Expert Data Pooled', color='#1f77b4', rotation=270, labelpad=25)
    ax_twin.tick_params(axis='y', labelcolor='#1f77b4')
    ax_twin.set_ylim(0, 105)
    
    legend_elements = [
        Line2D([0], [0], color='black', linewidth=3.0, label=r'MA-BC Pareto Dist. ($L_\infty$)'),
        Line2D([0], [0], label=r'Isolated BC', **iso_style),
        Line2D([0], [0], label=r'Naive BC', **naive_style),
        Line2D([0], [0], color='#d62728', linestyle='-.', linewidth=2.5, label=r'Empirical Action Bias ($L_2$)'),
        Patch(facecolor='#1f77b4', edgecolor='#1f77b4', alpha=0.2, linewidth=2.0, label=r'% Data Pooled')
    ]
    
    fig.legend(handles=legend_elements, loc='lower center', bbox_to_anchor=(0.5, 0.01), 
               ncol=3, framealpha=0.95, fontsize=16, columnspacing=2.0)
    
    plt.subplots_adjust(bottom=0.28, left=0.06, right=0.94, top=0.90)
    
    plt.savefig(filename, dpi=300) 
    plt.show()

In [ ]:
res_k1, _ = run_drone_sweep(pf_costs, A, B, Q1, Q2, R, n_sizes, delta_sweeps)

P1_opt = solve_discrete_are(A, B, Q1, R)
K1_true = np.linalg.inv(R + B.T @ P1_opt @ B) @ (B.T @ P1_opt @ A)
P2_opt = solve_discrete_are(A, B, Q2, R)
K2_true = np.linalg.inv(R + B.T @ P2_opt @ B) @ (B.T @ P2_opt @ A)

delta_s_grid, res_ex1, err1_ind, err1_naive = get_agile_tradeoff_data(
    pf_costs, A, B, Q1, Q2, R, K1_true, K2_true, N=100
)

In [ ]:
plot_agile_expert_summary(
    n_sizes, res_k1, delta_sweeps, 
    delta_s_grid, res_ex1, err1_ind, err1_naive,
    filename="drone_agile_summary.pdf"
)

# Economic Expert Only

In [ ]:
def get_eco_tradeoff_data(pf_costs, A, B, Q1, Q2, R, K1_true, K2_true, N=100):
    print(f"Running empirical bias-variance sweep for Economic Expert at N={N}...")
    np.random.seed(42)
    
    X1, U1 = generate_expert_rollouts(N, K1_true, A, B)
    X2, U2 = generate_expert_rollouts(N, K2_true, A, B)
    
    K2_indep = fit_independent(X2, U2)
    err2_indep = get_linf_distance_to_pf(K2_indep, pf_costs, Q1, Q2, R, A, B)
    
    K_naive = fit_pooled(X1, U1, X2, U2)
    err2_naive = get_linf_distance_to_pf(K_naive, pf_costs, Q1, Q2, R, A, B)
    
    delta_s_grid = np.logspace(-2, 0, 50) 
    results_ex2 = {'pct': [], 'bias': [], 'err': []}
    
    for ds in delta_s_grid:
        X2_aug, U2_aug = get_augmented_dataset_minmax(X2, U2, X1, U1, ds, K_base=K2_true, optimistic=False)
        num_pooled2 = len(X2_aug) - len(X2)
        results_ex2['pct'].append((num_pooled2 / len(X1)) * 100.0)
        
        if num_pooled2 > 0: 
            bias2 = np.mean(np.linalg.norm(U2_aug[len(U2):] - (-X2_aug[len(X2):] @ K2_true.T), axis=1))
        else: 
            bias2 = 0.0
            
        results_ex2['bias'].append(bias2)
        K2_hat = fit_independent(X2_aug, U2_aug)
        err2 = get_linf_distance_to_pf(K2_hat, pf_costs, Q1, Q2, R, A, B)
        results_ex2['err'].append(err2 if not np.isnan(err2) else 1e4)

    return delta_s_grid, results_ex2, err2_indep, err2_naive

In [ ]:
def plot_eco_expert_summary(n_sizes, res_k2, delta_s_list, 
                            delta_s_grid, results_ex2, err2_indep, err2_naive, 
                            filename="drone_eco_summary.pdf"):
    fig = plt.figure(figsize=(10, 5.2))
    
    gs = gridspec.GridSpec(1, 4, width_ratios=[1, 0.03, 0.20, 1.15], wspace=0.15)
    ax1 = fig.add_subplot(gs[0])
    cax = fig.add_subplot(gs[1])
    ax2 = fig.add_subplot(gs[3])
    
    iso_style = {'color': 'orange', 'linestyle': '--', 'linewidth': 2.5, 'marker': 'o', 'markersize': 8}
    naive_style = {'color': '#d62728', 'linestyle': '--', 'linewidth': 2.5, 'marker': 'X', 'markersize': 8}
    
    cmap = cm.viridis
    norm = mcolors.Normalize(vmin=min(delta_s_list), vmax=max(delta_s_list))
    sm = cm.ScalarMappable(cmap=cmap, norm=norm); sm.set_array([])
    
    for ds in delta_s_list:
        key = rf'MA-BC ($\delta_s={ds}$)'
        if key in res_k2:
            ax1.plot(n_sizes, res_k2[key], color=cmap(norm(ds)), linewidth=1.5, alpha=0.4, zorder=2)
            
    ax1.plot(n_sizes, res_k2['Naive Pool'], zorder=10, **naive_style)
    ax1.plot(n_sizes, res_k2['Independent'], zorder=10, **iso_style)
    
    ax1.set_yscale('log')
    ax1.set_xscale('log')
    ax1.grid(True, ls=":", alpha=0.6)
    
    ax1.set_title('Sample Efficiency')
    ax1.set_ylabel(r"$L_\infty$ Distance to PF")
    ax1.set_xlabel(r"Dataset Size per Expert")
    
    fig.colorbar(sm, cax=cax).set_label(r'Sharing Threshold ($\delta_s$)', labelpad=15)
    
    ax_twin = ax2.twinx()  
    
    ax2.plot(delta_s_grid, results_ex2['err'], '-', color='black', linewidth=3.0)
    
    ax2.plot(delta_s_grid, [err2_indep]*len(delta_s_grid), markevery=5, zorder=10, **iso_style)
    ax2.plot(delta_s_grid, [err2_naive]*len(delta_s_grid), markevery=5, zorder=10, **naive_style) 
    
    ax2.plot(delta_s_grid, results_ex2['bias'], '-.', color='#d62728', linewidth=2.5)
    
    ax_twin.fill_between(delta_s_grid, 0, results_ex2['pct'], color='#1f77b4', alpha=0.15)
    ax_twin.plot(delta_s_grid, results_ex2['pct'], '-', color='#1f77b4', linewidth=2.0)
    
    ax2.set_yscale('log')
    ax2.set_xscale('log')
    ax2.grid(True, ls=':', alpha=0.6)
    ax2.set_title(r'Empirical Bias-Data Tradeoff | $N_1=N_2=100$', pad=15)
    ax2.set_xlabel(r'Sharing Threshold ($\delta_s$)')
    ax2.set_ylabel("Error Metric")
    
    ax_twin.set_ylabel('% Other Expert Data Pooled', color='#1f77b4', rotation=270, labelpad=25)
    ax_twin.tick_params(axis='y', labelcolor='#1f77b4')
    ax_twin.set_ylim(0, 105)
    
    legend_elements = [
        Line2D([0], [0], label=r'Isolated BC (Independent Datasets)', **iso_style),
        Line2D([0], [0], label=r'Naive BC (Joint Dataset)', **naive_style),
        Line2D([0], [0], color='#d62728', linestyle='-.', linewidth=2.5, label=r'Empirical Action Bias ($\|A_{\mathrm{pooled}} - A_{\mathrm{true}}\|_2$)'),
        Line2D([0], [0], color='black', linewidth=3.0, label=r'MA-BC Pareto Dist. ($L_\infty$)'),
        Patch(facecolor='#1f77b4', edgecolor='#1f77b4', alpha=0.2, linewidth=2.0, label=r'% Data Pooled')
    ]
    
    fig.legend(handles=legend_elements, loc='lower center', bbox_to_anchor=(0.5, 0.01), 
               ncol=3, framealpha=0.95, columnspacing=2.0)
    
    plt.subplots_adjust(bottom=0.28, left=0.06, right=0.94, top=0.90)
    
    plt.savefig(filename, dpi=300) 
    plt.show()

In [ ]:
P1_opt = solve_discrete_are(A, B, Q1, R)
K1_true = np.linalg.inv(R + B.T @ P1_opt @ B) @ (B.T @ P1_opt @ A)
P2_opt = solve_discrete_are(A, B, Q2, R)
K2_true = np.linalg.inv(R + B.T @ P2_opt @ B) @ (B.T @ P2_opt @ A)

delta_s_grid, res_ex2, err2_ind, err2_naive = get_eco_tradeoff_data(
    pf_costs, A, B, Q1, Q2, R, K1_true, K2_true, N=100
)

In [ ]:
plot_eco_expert_summary(
    n_sizes, res_k2, delta_sweeps, 
    delta_s_grid, res_ex2, err2_ind, err2_naive,
    filename="drone_eco_summary.pdf"
)

In [ ]:
def plot_eco_expert_summary(n_sizes, res_k2, delta_s_list, 
                            delta_s_grid, 
                            results_ex2_40, err2_indep_40, err2_naive_40, 
                            results_ex2_100, err2_indep_100, err2_naive_100, 
                            filename="drone_eco_summary.pdf"):
    
    fig = plt.figure(figsize=(19, 5.5))
    
    gs = gridspec.GridSpec(1, 6, width_ratios=[1.0, 0.02, 0.40, 1.0, 0.18, 1.0], wspace=0.0)
    
    ax1 = fig.add_subplot(gs[0])
    cax = fig.add_subplot(gs[1])
    ax2 = fig.add_subplot(gs[3])
    ax3 = fig.add_subplot(gs[5])
    
    iso_style = {'color': 'orange', 'linestyle': '--', 'linewidth': 2.5, 'marker': 'o', 'markersize': 8}
    naive_style = {'color': '#d62728', 'linestyle': '--', 'linewidth': 2.5, 'marker': 'X', 'markersize': 8}
    
    cmap = cm.viridis
    norm = mcolors.Normalize(vmin=min(delta_s_list), vmax=max(delta_s_list))
    sm = cm.ScalarMappable(cmap=cmap, norm=norm); sm.set_array([])
    
    for ds in delta_s_list:
        key = rf'MA-BC ($\delta_s={ds}$)'
        if key in res_k2:
            ax1.plot(n_sizes, res_k2[key], color=cmap(norm(ds)), linewidth=1.5, alpha=0.4, zorder=2)
            
    ax1.plot(n_sizes, res_k2['Naive Pool'], zorder=10, **naive_style)
    ax1.plot(n_sizes, res_k2['Independent'], zorder=10, **iso_style)
    
    ax1.set_yscale('log')
    ax1.set_xscale('log')
    ax1.grid(True, ls=":", alpha=0.6)
    
    ax1.set_title('Sample Efficiency', pad=15)
    ax1.set_ylabel(r"$L_\infty$ Distance to PF")
    ax1.set_xlabel(r"Dataset Size per Expert ($N$)")
    
    fig.colorbar(sm, cax=cax).set_label(r'Sharing Threshold ($\delta_s$)', labelpad=15)
    
    ax_twin2 = ax2.twinx()  
    
    ax2.plot(delta_s_grid, results_ex2_40['err'], '-', color='black', linewidth=3.0)
    ax2.plot(delta_s_grid, [err2_indep_40]*len(delta_s_grid), markevery=5, zorder=10, **iso_style)
    ax2.plot(delta_s_grid, [err2_naive_40]*len(delta_s_grid), markevery=5, zorder=10, **naive_style) 
    ax2.plot(delta_s_grid, results_ex2_40['bias'], '-.', color='#d62728', linewidth=2.5)
    
    ax_twin2.fill_between(delta_s_grid, 0, results_ex2_40['pct'], color='#1f77b4', alpha=0.15)
    ax_twin2.plot(delta_s_grid, results_ex2_40['pct'], '-', color='#1f77b4', linewidth=2.0)
    
    ax2.set_yscale('log')
    ax2.set_xscale('log')
    ax2.grid(True, ls=':', alpha=0.6)
    ax2.set_title(r'Empirical Bias-Data Tradeoff | $N_1=N_2=40$', pad=15)
    ax2.set_xlabel(r'Sharing Threshold ($\delta$)')
    ax2.set_ylabel("Error Metric")
    
    ax_twin2.tick_params(axis='y', labelcolor='#1f77b4')
    ax_twin2.set_ylim(0, 105)
    
    ax_twin3 = ax3.twinx()  
    
    ax3.plot(delta_s_grid, results_ex2_100['err'], '-', color='black', linewidth=3.0)
    ax3.plot(delta_s_grid, [err2_indep_100]*len(delta_s_grid), markevery=5, zorder=10, **iso_style)
    ax3.plot(delta_s_grid, [err2_naive_100]*len(delta_s_grid), markevery=5, zorder=10, **naive_style) 
    ax3.plot(delta_s_grid, results_ex2_100['bias'], '-.', color='#d62728', linewidth=2.5)
    
    ax_twin3.fill_between(delta_s_grid, 0, results_ex2_100['pct'], color='#1f77b4', alpha=0.15)
    ax_twin3.plot(delta_s_grid, results_ex2_100['pct'], '-', color='#1f77b4', linewidth=2.0)
    
    ax3.set_yscale('log')
    ax3.set_xscale('log')
    ax3.grid(True, ls=':', alpha=0.6)
    ax3.set_title(r'Empirical Bias-Data Tradeoff | $N_1=N_2=100$', pad=15)
    ax3.set_xlabel(r'Sharing Threshold ($\delta$)')
    
    ax3.set_ylabel("") 
    
    ax_twin3.set_ylabel('% Other Expert Data Pooled', color='#1f77b4', rotation=270, labelpad=25)
    ax_twin3.tick_params(axis='y', labelcolor='#1f77b4')
    ax_twin3.set_ylim(0, 105)
    
    legend_elements = [
        Line2D([0], [0], color='black', linewidth=3.0, label=r'MA-BC Pareto Dist. ($L_\infty$)'),
        Line2D([0], [0], label=r'Isolated BC (Independent Datasets)', **iso_style),
        Line2D([0], [0], label=r'Naive BC (Joint Dataset)', **naive_style),
        Line2D([0], [0], color='#d62728', linestyle='-.', linewidth=2.5, label=r'Empirical Action Bias ($\|A_{\mathrm{pooled}} - A_{\mathrm{true}}\|_2$)'),
        Patch(facecolor='#1f77b4', edgecolor='#1f77b4', alpha=0.2, linewidth=2.0, label=r'% Data Pooled')
    ]
    
    fig.legend(handles=legend_elements, loc='lower center', bbox_to_anchor=(0.5, 0.01), 
               ncol=5, framealpha=0.95, columnspacing=1.5)
    
    plt.subplots_adjust(bottom=0.25, left=0.04, right=0.96, top=0.90)
    
    plt.savefig(filename, bbox_inches=None, dpi=300, transparent=True) 
    plt.show()


P1_opt = solve_discrete_are(A, B, Q1, R)
K1_true = np.linalg.inv(R + B.T @ P1_opt @ B) @ (B.T @ P1_opt @ A)
P2_opt = solve_discrete_are(A, B, Q2, R)
K2_true = np.linalg.inv(R + B.T @ P2_opt @ B) @ (B.T @ P2_opt @ A)

delta_s_grid_40, res_ex2_40, err2_ind_40, err2_naive_40 = get_eco_tradeoff_data(
    pf_costs, A, B, Q1, Q2, R, K1_true, K2_true, N=40
)

delta_s_grid_100, res_ex2_100, err2_ind_100, err2_naive_100 = get_eco_tradeoff_data(
    pf_costs, A, B, Q1, Q2, R, K1_true, K2_true, N=100
)

plot_eco_expert_summary(
    n_sizes, res_k2, delta_sweeps, 
    delta_s_grid_40, 
    res_ex2_40, err2_ind_40, err2_naive_40,
    res_ex2_100, err2_ind_100, err2_naive_100,
    filename="drone_eco_summary.pdf"
)